In [1]:
from pyspark.sql import SparkSession
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

spark = SparkSession.builder \
    .appName("Steam_ML_Predictor") \
    .config("spark.driver.memory", "8g") \
    .getOrCreate()

print("Loading Processed Data for ML...")

df = spark.read.parquet("data/processed/steam_transformed.parquet")

df_ml = df.dropna(subset=["hours", "price_final", "is_positive"])

assembler = VectorAssembler(
    inputCols=["hours", "price_final"],
    outputCol="features"
)

df_assembled = assembler.transform(df_ml)
df_model_ready = df_assembled.select("features", "is_positive")

train_data, test_data = df_model_ready.randomSplit([0.8, 0.2], seed=42)
print(f"Training Data Count: {train_data.count()}")
print(f"Testing Data Count: {test_data.count()}")

print("Training Logistic Regression Model...")
lr = LogisticRegression(featuresCol="features", labelCol="is_positive", maxIter=10)
lr_model = lr.fit(train_data)

predictions = lr_model.transform(test_data)

evaluator = MulticlassClassificationEvaluator(
    labelCol="is_positive", 
    predictionCol="prediction", 
    metricName="accuracy"
)
accuracy = evaluator.evaluate(predictions)

print("-" * 30)
print(f"Model Accuracy: {accuracy * 100:.2f}%")
print("-" * 30)

predictions.select("features", "is_positive", "prediction", "probability").show(5)

Loading Processed Data for ML...
Training Data Count: 32921884
Testing Data Count: 8232910
Training Logistic Regression Model...
------------------------------
Model Accuracy: 85.78%
------------------------------
+---------+-----------+----------+--------------------+
| features|is_positive|prediction|         probability|
+---------+-----------+----------+--------------------+
|(2,[],[])|          0|       1.0|[0.15310044471611...|
|(2,[],[])|          0|       1.0|[0.15310044471611...|
|(2,[],[])|          0|       1.0|[0.15310044471611...|
|(2,[],[])|          0|       1.0|[0.15310044471611...|
|(2,[],[])|          0|       1.0|[0.15310044471611...|
+---------+-----------+----------+--------------------+
only showing top 5 rows

